
# Bank Marketing Classification

## Project Overview

This project predicts whether a bank client will subscribe to a term deposit using machine learning classification models.


## 1. Import Libraries and Load Data

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

display(train.head())

Train shape: (32940, 22)
Test shape: (8236, 20)


,id,age,job,marital,education,default,housing,loan,contact,month,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,0,39,management,single,basic.9y,unknown,yes,no,telephone,may,...,3,999,0,nonexistent,1.1,93.994,-36.4,4.856,5191.0,no
1,1,32,admin.,single,university.degree,no,yes,no,cellular,aug,...,2,999,0,nonexistent,1.4,93.444,-36.1,4.963,5228.1,no
2,2,41,technician,married,high.school,no,no,yes,cellular,aug,...,1,999,0,nonexistent,1.4,93.444,-36.1,4.963,5228.1,no
3,3,39,admin.,married,professional.course,no,yes,yes,telephone,may,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,4,51,blue-collar,married,basic.9y,unknown,no,no,telephone,jun,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.864,5228.1,no


## 2. Data Inspection

In [10]:

print("Columns:")
print(train.columns.tolist())

print("\nMissing values:")
print(train.isnull().sum())

print("\nTarget distribution:")
print(train["y"].value_counts())
print(train["y"].value_counts(normalize=True) * 100)


Columns:
['id', 'age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']

Missing values:
id                0
age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

Target distribution:
y
no     29229
yes     3711
Name: count, dtype: int64
y
no     88.734062
yes    11.265938
Name: proportion, dtype: float64


## 3. Data Preprocessing

In [11]:


print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nColumns missing from test:")
print(sorted(set(train.columns) - set(test.columns)))

print("\nColumns missing from train:")
print(sorted(set(test.columns) - set(train.columns)))

print("\nMissing values in train:")
print(train.isnull().sum())

print("\nTarget distribution:")
print(train["y"].value_counts())
print((train["y"].value_counts(normalize=True) * 100).round(2))


Train shape: (32940, 22)
Test shape: (8236, 20)

Columns missing from test:
['duration', 'y']

Columns missing from train:
[]

Missing values in train:
id                0
age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

Target distribution:
y
no     29229
yes     3711
Name: count, dtype: int64
y
no     88.73
yes    11.27
Name: proportion, dtype: float64


## 4. Model Training

In [12]:

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import numpy as np


SEED = 2650

X = train.drop(columns=["y", "id", "duration"])
y = train["y"].map({"no": 0, "yes": 1})

X_test = test.drop(columns=["id"])

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numerical_features = X.select_dtypes(exclude=["object"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numerical_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(
        random_state=SEED, class_weight="balanced", max_depth=10
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=SEED,
        class_weight="balanced", n_jobs=-1
    )
}

results = []
trained_models = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_valid)
    probabilities = pipeline.predict_proba(X_valid)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_valid, predictions),
        "Precision": precision_score(y_valid, predictions, zero_division=0),
        "Recall": recall_score(y_valid, predictions, zero_division=0),
        "F1": f1_score(y_valid, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_valid, probabilities)
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
display(results_df.round(4))


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.8365,0.3701,0.6429,0.4697,0.8028
2,Random Forest,0.8978,0.5873,0.3127,0.4081,0.7899
1,Decision Tree,0.8555,0.4004,0.5687,0.4699,0.7447


## 5. Model Evaluation

In [13]:

import joblib
from pathlib import Path

best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

Path("outputs").mkdir(exist_ok=True)

joblib.dump(best_model, "outputs/model.joblib")

print("Best model:", best_model_name)
print("Model saved to outputs/model.joblib")


Best model: Logistic Regression
Model saved to outputs/model.joblib


## 6. Best Model Selection

In [14]:

test_predictions = best_model.predict(X_test)
test_probabilities = best_model.predict_proba(X_test)[:, 1]

predictions_df = pd.DataFrame({
    "id": test["id"],
    "y": np.where(test_predictions == 1, "yes", "no")
})

predictions_df.to_csv("outputs/predictions.csv", index=False)

print("Predictions saved to outputs/predictions.csv")
print("Prediction shape:", predictions_df.shape)
display(predictions_df.head())


Predictions saved to outputs/predictions.csv
Prediction shape: (8236, 2)


,id,y
0,32940,no
1,32941,no
2,32942,no
3,32943,no
4,32944,no


## 7. Test Predictions

In [15]:

results_df.to_csv("outputs/model_comparison.csv", index=False)

print("Model comparison saved successfully.")
display(results_df)


Model comparison saved successfully.


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.836521,0.370054,0.642857,0.469719,0.802756
2,Random Forest,0.897845,0.587342,0.312668,0.408091,0.789884
1,Decision Tree,0.855495,0.400380,0.568733,0.469933,0.744730


In [16]:

print("Model file exists:", Path("outputs/model.joblib").exists())
print("Predictions file exists:", Path("outputs/predictions.csv").exists())
print("Comparison file exists:", Path("outputs/model_comparison.csv").exists())

predictions_check = pd.read_csv("outputs/predictions.csv")

print("Prediction rows:", len(predictions_check))
print("Expected rows:", len(test))
print("Missing predictions:", predictions_check["y"].isnull().sum())
print("Unique labels:", predictions_check["y"].unique())


Model file exists: True
Predictions file exists: True
Comparison file exists: True
Prediction rows: 8236
Expected rows: 8236
Missing predictions: 0
Unique labels: ['no' 'yes']


In [17]:
import sys
print(sys.executable)

c:\Users\Сымбат\Desktop\olzhabekova_ML_task_3\.venv\Scripts\python.exe
